# Construcción de la base por hogar (una fila por hogar)

**Entrada:** las tablas limpias de `02_limpieza_con_trazabilidad` (`data/processed/tablas_limpias`).
**Salida:** `data/processed/dataset_final.csv` (una fila por hogar) y `data/processed/diccionario_dataset_final.csv` (una fila por columna, con su significado). Es el dataset del proyecto: aquí termina la **construcción** y desde el cuaderno 04 empieza el proceso estándar de modelado.

**Por qué una fila por hogar:** la unidad de estudio es el hogar. Las personas (módulos 200 y 300) se resumen a hogar con conteos y proporciones; si no, el mismo hogar aparecería varias veces y la partición por conglomerado perdería sentido.

**Cada columna cae en uno de estos grupos:**
- **predictor:** entra al modelo. Solo características observables del hogar, de la vivienda y de las personas.
- **identificador / ponderación:** sirven para unir, dividir por conglomerado y ponderar, no para predecir.
- **objetivo:** `y` (1 = pobre extremo o no extremo; 0 = no pobre) y variantes para robustez.
- **solo análisis:** se usan después para ver a quién clasifica mal el modelo (por departamento, lengua, programa social), pero **nunca como predictor**.

## Paso 0. Configuración

**Por qué:** una función `num` convierte texto a número aceptando la coma decimal. Los CSV del INEI traen decimales con coma (`87,2856903076172`) y `pd.to_numeric` solo, los convierte en vacío. Así `factor07` no se pierde. Además cada columna creada se anota en un diccionario (`d(...)`).

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd().resolve().parent
LIMPIAS = RAIZ / "data" / "processed" / "tablas_limpias"
PROCESSED = RAIZ / "data" / "processed"


def cargar(nombre: str) -> pd.DataFrame:
    for ext, lector in ((".parquet", pd.read_parquet), (".pkl", pd.read_pickle)):
        ruta = LIMPIAS / f"{nombre}{ext}"
        if ruta.exists():
            return lector(ruta)
    raise FileNotFoundError(f"No encuentro {nombre} en {LIMPIAS}. Corre primero 02_limpieza_con_trazabilidad.")


def num(serie: pd.Series) -> pd.Series:
    """Texto -> número, aceptando coma decimal. Lo que no sea número queda vacío."""
    return pd.to_numeric(serie.astype(str).str.replace(",", ".", regex=False), errors="coerce")


def presentes(df: pd.DataFrame, columnas):
    faltan = [c for c in columnas if c not in df.columns]
    if faltan:
        print(f"  Aviso: no están en los datos y se omiten: {faltan}")
    return [c for c in columnas if c in df.columns]


DICC = []   # diccionario de la base final


def d(variable, grupo, origen, construccion):
    DICC.append({"variable": variable, "grupo": grupo, "origen": origen, "construccion": construccion})


sumaria, m100, m200, m300 = (cargar(n) for n in ("sumaria", "m100", "m200", "m300"))
print({n: len(t) for n, t in zip(("sumaria", "m100", "m200", "m300"), (sumaria, m100, m200, m300))})

## Paso 1. Sumaria: identificadores, objetivo, ponderación y variables de análisis

**Por qué:**
- `y = 1` si `POBREZA` es 1 o 2 (según el diccionario: 1 pobre extremo, 2 pobre no extremo, 3 no pobre). También se guardan `y_extremo` y `y_pobre_o_vulnerable` (con `POBREZAV`) solo para pruebas de robustez.
- `factor07` es el factor de expansión. No se usa como predictor, solo para ponderar si se quiere.
- `departamento`, `dominio` y `estrato` se guardan **solo para análisis**: sirven para revisar si el modelo falla más en una región, no para decidir quién recibe ayuda.
- `ingtpu01` e `ingtpu03` son transferencias públicas (Juntos y Pensión 65). Si el hogar ya las recibe, su ingreso incluye ayuda del propio programa que queremos focalizar, así que **no entran como predictor**. Se guardan como marcas 0/1 solo para análisis.
- `mieperho` (tamaño del hogar) y `area_urbana` (`estrato` 1 a 5) sí son predictores: se observan sin medir el gasto. El criterio de área urbana es el habitual del INEI; confírmalo en la documentación.

In [ ]:
print("PASO 1. SUMARIA")
s = sumaria.copy()
for c in presentes(s, ["pobreza", "pobrezav", "mieperho", "factor07", "estrato", "dominio", "ingtpu01", "ingtpu03"]):
    s[c] = num(s[c])

base = pd.DataFrame({"id_hogar": s["id_hogar"].values})
base["conglome"] = s["conglome"].values
base["ubigeo"] = s["ubigeo"].values
base["factor07"] = s["factor07"].values
base["pobreza_cod"] = s["pobreza"].values
base["y"] = s["pobreza"].isin([1, 2]).astype(int).values
base["y_extremo"] = (s["pobreza"] == 1).astype(int).values
if "pobrezav" in s.columns:
    base["y_pobre_o_vulnerable"] = s["pobrezav"].isin([1, 2, 3]).astype(int).values

# solo análisis
base["departamento"] = s["ubigeo"].str[:2].values
base["dominio"] = s["dominio"].values
base["estrato"] = s["estrato"].values
if "ingtpu01" in s.columns:
    base["recibe_juntos"] = (s["ingtpu01"] > 0).astype(int).values
if "ingtpu03" in s.columns:
    base["recibe_pension65"] = (s["ingtpu03"] > 0).astype(int).values

# predictores
base["mieperho"] = s["mieperho"].values
base["area_urbana"] = (s["estrato"] <= 5).astype(int).values

d("id_hogar", "identificador", "sumaria", "CONGLOME + VIVIENDA + HOGAR (texto con ceros)")
d("conglome", "identificador", "sumaria", "Conglomerado: se usa para dividir entrenamiento/prueba por grupos")
d("ubigeo", "identificador", "sumaria", "Código de ubicación (6 dígitos)")
d("factor07", "ponderacion", "sumaria", "Factor de expansión, convertido a número (coma decimal)")
d("pobreza_cod", "objetivo", "sumaria", "POBREZA original: 1 pobre extremo, 2 pobre no extremo, 3 no pobre")
d("y", "objetivo", "sumaria", "1 si POBREZA es 1 o 2; 0 si es 3")
d("y_extremo", "objetivo", "sumaria", "1 si POBREZA = 1 (robustez)")
d("y_pobre_o_vulnerable", "objetivo", "sumaria", "1 si POBREZAV es 1, 2 o 3 (robustez)")
d("departamento", "solo_analisis", "sumaria", "Primeros 2 dígitos de UBIGEO")
d("dominio", "solo_analisis", "sumaria", "DOMINIO original")
d("estrato", "solo_analisis", "sumaria", "ESTRATO original (tamaño de centro poblado)")
d("recibe_juntos", "solo_analisis", "sumaria", "1 si INGTPU01 > 0 (transferencia: no se usa como predictor)")
d("recibe_pension65", "solo_analisis", "sumaria", "1 si INGTPU03 > 0 (transferencia: no se usa como predictor)")
d("mieperho", "predictor", "sumaria", "Miembros del hogar (definición INEI)")
d("area_urbana", "predictor", "sumaria", "1 si ESTRATO <= 5")
print(f"  Hogares: {len(base):,} | pobres: {base['y'].sum():,} ({base['y'].mean():.1%}) | factor07 vacío: {base['factor07'].isna().mean():.2%}")

## Paso 2. Módulo 200: de personas a hogar (composición y jefe)

**Por qué:** la estructura del hogar (cuántos niños, cuántos adultos mayores, quién es el jefe) es de lo más usado para identificar hogares vulnerables, y se observa sin preguntar por el gasto. Los programas sociales miran justamente eso: niños pequeños y adultos mayores.

- Edad: `P208A` (años); si está vacía y hay `P208B` (meses), el niño tiene menos de 1 año y se registra con edad 0. El código 99 es "sin dato" y se convierte en vacío.
- Dependientes: menores de 15 y de 65 años o más (definición demográfica estándar).
- Conteos (`n_*`) en vez de proporciones donde conviene: un hogar sin niños tiene 0 niños, no un dato perdido.
- Jefe del hogar: `P203 = 1`.

In [ ]:
print("PASO 2. MODULO 200 (personas -> hogar)")
p = m200.copy()
for c in presentes(p, ["p203", "p207", "p208a", "p208b", "p209"]):
    p[c] = num(p[c])

edad = p["p208a"].where(p["p208a"] < 99)
if "p208b" in p.columns:
    edad = edad.mask(edad.isna() & p["p208b"].between(1, 11), 0)
p["edad"] = edad
p["m5"] = (p["edad"] < 5).astype(int)
p["n6_17"] = p["edad"].between(6, 17).astype(int)
p["a18"] = (p["edad"] >= 18).astype(int)
p["m65"] = (p["edad"] >= 65).astype(int)
p["dep"] = ((p["edad"] < 15) | (p["edad"] >= 65)).astype(int)
p["mujer"] = (p["p207"] == 2).astype(int)

agg = p.groupby("id_hogar").agg(
    n_menores_5=("m5", "sum"), n_ninos_6_17=("n6_17", "sum"), n_adultos_18_mas=("a18", "sum"),
    n_mayores_65=("m65", "sum"), prop_dependientes=("dep", "mean"), prop_mujeres=("mujer", "mean"),
    edad_media=("edad", "mean"))

jefe = p[p["p203"] == 1].drop_duplicates("id_hogar").set_index("id_hogar")
jefe_f = pd.DataFrame({"jefe_edad": jefe["edad"], "jefe_mujer": jefe["mujer"], "jefe_estado_civil": jefe["p209"]})
print(f"  Hogares sin jefe identificado: {len(set(base['id_hogar']) - set(jefe_f.index)):,}")

for v, c in [("n_menores_5", "Personas menores de 5 años"), ("n_ninos_6_17", "Personas de 6 a 17 años"),
             ("n_adultos_18_mas", "Personas de 18 años o más"), ("n_mayores_65", "Personas de 65 años o más"),
             ("prop_dependientes", "Proporción de menores de 15 y mayores de 64"), ("prop_mujeres", "Proporción de mujeres"),
             ("edad_media", "Edad media de los miembros"), ("jefe_edad", "Edad del jefe(a)"),
             ("jefe_mujer", "1 si el jefe(a) es mujer"), ("jefe_estado_civil", "P209 del jefe(a) (categoría)")]:
    d(v, "predictor", "m200", c)

## Paso 3. Módulo 300: educación, lectura, asistencia escolar e internet

**Por qué:** el nivel educativo del hogar es uno de los mejores predictores de pobreza y se observa fácilmente. El módulo 300 no trae la edad, así que se une con el módulo 200 por `id_persona`.

- Educación de adultos: se consideran personas de 15 años o más con nivel conocido. `P301A` = 12 (básica especial) y 99 no son ordenables y quedan vacíos.
- **Lectura y asistencia como conteos** (`n_adultos_no_leen`, `n_ninos_6_17_no_asisten`): en tu primer intento eran proporciones con 61 a 67 % de vacíos, porque muchos hogares no tienen esas personas. Con conteos, "no hay nadie en esa situación" vale 0 y deja de ser un dato perdido.
- `P302` y `P307` traen 9 = "sin dato"; solo se cuentan respuestas 1 y 2.
- **Lengua materna del jefe** (`P300A`): solo para análisis. Las lenguas indígenas son los códigos 1, 2, 3 y 10 a 15 del diccionario. No se usa como predictor para no decidir quién recibe ayuda según su lengua; sí sirve para ver si el modelo falla más con esos hogares.

In [ ]:
print("PASO 3. MODULO 300 (educacion)")
LENGUAS_INDIGENAS = {1, 2, 3, 10, 11, 12, 13, 14, 15}
cols300 = presentes(m300, ["p300a", "p301a", "p302", "p307", "p314a"])
e = m300[["id_persona", "id_hogar"] + cols300].copy()
for c in cols300:
    e[c] = num(e[c])
e = e.merge(p[["id_persona", "edad", "p203"]], on="id_persona", how="left")
e["nivel"] = e["p301a"].where(e["p301a"].between(1, 11))

adultos = e[(e["edad"] >= 15) & e["nivel"].notna()]
edu = pd.DataFrame({
    "edu_max_adultos": adultos.groupby("id_hogar")["nivel"].max(),
    "prop_adultos_prim_inc": (adultos["nivel"] <= 3).groupby(adultos["id_hogar"]).mean(),   # hasta primaria incompleta
})
edu["n_adultos_no_leen"] = ((e["edad"] >= 15) & (e["p302"] == 2)).groupby(e["id_hogar"]).sum()
edu["n_ninos_6_17_no_asisten"] = (e["edad"].between(6, 17) & (e["p307"] == 2)).groupby(e["id_hogar"]).sum()
if "p314a" in e.columns:
    usa = e[e["p314a"].isin([1, 2])]
    edu["prop_usa_internet"] = (usa["p314a"] == 1).groupby(usa["id_hogar"]).mean()

jefe_e = e[e["p203"] == 1].drop_duplicates("id_hogar").set_index("id_hogar")
edu["jefe_nivel_edu"] = jefe_e["nivel"]
lengua = jefe_e["p300a"].where(jefe_e["p300a"] < 99)
edu["jefe_lengua_materna"] = lengua
edu["jefe_lengua_indigena"] = lengua.isin(LENGUAS_INDIGENAS).astype(float).where(lengua.notna())

for v, g, c in [
    ("edu_max_adultos", "predictor", "Máximo nivel educativo (P301A, 1 a 11) entre miembros de 15 años o más"),
    ("prop_adultos_prim_inc", "predictor", "Proporción de adultos (15+) con hasta primaria incompleta (nivel <= 3)"),
    ("n_adultos_no_leen", "predictor", "Adultos (15+) que no saben leer ni escribir (P302 = 2); 0 si no hay"),
    ("n_ninos_6_17_no_asisten", "predictor", "Personas de 6 a 17 años que no asisten a la escuela (P307 = 2); 0 si no hay"),
    ("prop_usa_internet", "predictor", "Proporción de miembros que usó internet el mes anterior (P314A); vacío si no se preguntó"),
    ("jefe_nivel_edu", "predictor", "Nivel educativo del jefe(a) (P301A, 1 a 11)"),
    ("jefe_lengua_materna", "solo_analisis", "P300A del jefe(a)"),
    ("jefe_lengua_indigena", "solo_analisis", "1 si la lengua materna del jefe(a) es indígena (códigos 1, 2, 3, 10 a 15)")]:
    if v in edu.columns:
        d(v, g, "m300 + m200", c)

## Paso 4. Módulo 100: vivienda y hogar

**Por qué:** material de paredes y techo, agua, desagüe, electricidad, combustible para cocinar y bienes del hogar describen las condiciones de vida y se pueden preguntar sin conocer el gasto. Son la parte "clásica" de un modelo de focalización.

Las columnas se toman de tu lista (sin montos: no hay `D*` ni `I*`).
- **Binarias** (0 = pase, 1 = sí): se dejan en 0/1.
- **Categóricas:** se dejan como código; más adelante se codifican con One-Hot.
- **Numéricas:** `P104` (habitaciones) y `P104A` (dormitorios).
- **Códigos de "sin dato"** (9 en las categóricas indicadas, 99 en las numéricas) pasan a vacío: si no, el modelo los leería como una categoría o un número grande real.
- **"No aplica" por salto de preguntas.** `P106A` (título de propiedad) solo se pregunta a hogares con vivienda propia: queda vacía exactamente cuando `P105A` es 1, 5, 6 o 7 (alquilada, cedida u otra forma). `P106B` (título en SUNARP) solo se pregunta a quien tiene título (`P106A = 1`). En esos casos el vacío no es un dato perdido, así que se recodifica a **0 = "no aplica"**. - `P114B1`, `P114B2` y `P114B3` (tipo de conexión a internet: fija, móvil pospago, móvil prepago) solo se preguntan si el hogar tiene internet (`P1144 = 1`). En tus datos están vacías exactamente cuando `P1144 = 0` (3 310 hogares), así que también pasan a **0 = "no aplica"**.

Los vacíos que quedan sí son datos perdidos (por ejemplo, un propietario sin respuesta) y se imputarán más adelante solo con el conjunto de entrenamiento.

In [ ]:
print("PASO 4. MODULO 100 (vivienda)")
M100_BINARIAS = [
    "p25_1", "p25_2", "p25_3", "p25_4", "p25_5",
    "p1121", "p1123", "p1124", "p1125", "p1126", "p1127",
    "p1131", "p1132", "p1133", "p1135", "p1136", "p1139", "p1137", "p1138",
    "p1141", "p1142", "p1143", "p1144", "p114b1", "p114b2", "p114b3", "p1145", "p1146",
]
M100_CATEGORICAS = ["p24a", "p24b", "p101", "p102", "p103", "p103a", "p105a", "p106a", "p106b",
                    "p110c", "p112a", "p113a", "t110", "t111a"]
M100_NUMERICAS = ["p104", "p104a"]
CODIGOS_PERDIDOS_9 = ["p24a", "p24b", "p105a", "p106a", "p106b", "p110c"]
CODIGOS_PERDIDOS_99 = ["p104", "p104a"]

cols = presentes(m100, M100_BINARIAS + M100_CATEGORICAS + M100_NUMERICAS)
v = m100[["id_hogar"] + cols].copy()
for c in cols:
    v[c] = num(v[c])
for c in M100_BINARIAS:
    if c in v.columns:
        v[c] = (v[c] > 0).astype(float).where(v[c].notna())
for c in CODIGOS_PERDIDOS_9:
    if c in v.columns:
        v[c] = v[c].where(v[c] != 9)
for c in CODIGOS_PERDIDOS_99:
    if c in v.columns:
        v[c] = v[c].where(v[c] != 99)

# "No aplica" por salto de preguntas -> código 0 (los vacíos restantes son datos perdidos reales)
if {"p105a", "p106a"} <= set(v.columns):
    n = int((v["p105a"].isin([1, 5, 6, 7]) & v["p106a"].isna()).sum())
    v.loc[v["p105a"].isin([1, 5, 6, 7]) & v["p106a"].isna(), "p106a"] = 0
    print(f"  p106a: {n:,} hogares no propietarios -> 0 (no aplica)")
if "p1144" in v.columns:
    for c in ["p114b1", "p114b2", "p114b3"]:
        if c in v.columns:
            m = (v["p1144"] == 0) & v[c].isna()
            v.loc[m, c] = 0
            print(f"  {c}: {int(m.sum()):,} hogares sin internet -> 0 (no aplica)")
if {"p106a", "p106b"} <= set(v.columns):
    n = int((v["p106a"].isin([0, 2, 3]) & v["p106b"].isna()).sum())
    v.loc[v["p106a"].isin([0, 2, 3]) & v["p106b"].isna(), "p106b"] = 0
    print(f"  p106b: {n:,} hogares sin título de propiedad -> 0 (no aplica)")

for c in cols:
    tipo = "binaria (0/1)" if c in M100_BINARIAS else ("numérica" if c in M100_NUMERICAS else "categórica (código)")
    perd = " | 9 -> vacío" if c in CODIGOS_PERDIDOS_9 else (" | 99 -> vacío" if c in CODIGOS_PERDIDOS_99 else "")
    if c in ("p106a", "p106b"):
        perd += " | 0 = no aplica (salto de preguntas)"
    if c in ("p114b1", "p114b2", "p114b3"):
        perd += " | 0 = no aplica (hogar sin internet, P1144 = 0)"
    d(c, "predictor", "m100", f"{tipo}{perd}")

## Paso 5. Unir todo a Sumaria y crear el hacinamiento

**Por qué:** la unión es un *left join* desde Sumaria, que define la población. Así nunca se agregan ni se pierden hogares, y se comprueba al final que siga habiendo 33 702.

- **Conteos sin personas = 0:** si un hogar no aparece en una tabla de conteos (por ejemplo, ningún niño que no asista), el conteo es 0. En las proporciones el vacío se mantiene.
- **Hacinamiento:** personas por dormitorio y por habitación. El 9,9 % de los hogares tiene 0 dormitorios (un solo ambiente para todo), que es justo el caso más hacinado; dejarlo vacío perdería esa señal. Por eso se divide entre `max(dormitorios, 1)` y se agrega la marca `sin_dormitorio`. Si el dato de `P104A` falta de verdad, el resultado queda vacío.

In [ ]:
print("PASO 5. UNION")
base = (base.merge(v, on="id_hogar", how="left")
            .merge(agg, left_on="id_hogar", right_index=True, how="left")
            .merge(jefe_f, left_on="id_hogar", right_index=True, how="left")
            .merge(edu, left_on="id_hogar", right_index=True, how="left"))

for c in ["n_adultos_no_leen", "n_ninos_6_17_no_asisten"]:
    base[c] = base[c].fillna(0)

base["sin_dormitorio"] = (base["p104a"] == 0).astype(float).where(base["p104a"].notna())
base["personas_por_dormitorio"] = base["mieperho"] / base["p104a"].clip(lower=1)
base["personas_por_habitacion"] = base["mieperho"] / base["p104"].where(base["p104"] > 0)
d("sin_dormitorio", "predictor", "m100", "1 si P104A = 0 (el hogar no tiene dormitorios)")
d("personas_por_dormitorio", "predictor", "sumaria + m100", "MIEPERHO / max(P104A, 1); vacío solo si P104A falta")
d("personas_por_habitacion", "predictor", "sumaria + m100", "MIEPERHO / P104 (vacío si P104 = 0 o falta)")
print(f"  Hogares sin dormitorio: {(base['sin_dormitorio'] == 1).mean():.1%}")

assert base["id_hogar"].is_unique and len(base) == len(sumaria), "la unión duplicó o perdió hogares"
print(f"  Base: {base.shape[0]:,} hogares x {base.shape[1]} columnas")

## Paso 6. Verificar

**Por qué:** antes de guardar se comprueban cuatro cosas: una fila por hogar, `factor07` sin vacíos, la proporción de pobres igual a la de Sumaria y el tamaño del hogar igual a los miembros contados. También se comprueba que **cada columna tenga su descripción en el diccionario**: si algo no está documentado, el cuaderno se detiene.

**Los vacíos no se llenan aquí.** Se conservan tal cual: llenarlos exige calcular medianas y modas, y eso debe hacerse solo con el conjunto de entrenamiento, después de partir (cuaderno 05).

In [ ]:
print("PASO 6. VERIFICACION")
assert base["factor07"].notna().all(), "factor07 tiene vacíos: revisa la conversión"
assert (base["mieperho"] == p.groupby("id_hogar").size().reindex(base["id_hogar"]).values).all(), "miembros contados != MIEPERHO"
print(f"  Hogares: {len(base):,} | pobres: {base['y'].sum():,} ({base['y'].mean():.1%})")

dicc = pd.DataFrame(DICC).drop_duplicates("variable")
grupos = dicc.set_index("variable")["grupo"]
en_base = list(base.columns)
sin_doc = [c for c in en_base if c not in grupos.index]
assert not sin_doc, f"columnas sin documentar en el diccionario: {sin_doc}"
dicc = dicc[dicc["variable"].isin(en_base)]

predictores = [c for c in en_base if grupos[c] == "predictor"]
binarias = [c for c in predictores if c in M100_BINARIAS or c in ("area_urbana", "jefe_mujer", "sin_dormitorio")]
categoricas = [c for c in predictores if c in M100_CATEGORICAS or c == "jefe_estado_civil"]
numericas = [c for c in predictores if c not in binarias and c not in categoricas]
print(f"  Predictores: {len(predictores)} (numéricos {len(numericas)}, binarios {len(binarias)}, categóricos {len(categoricas)})")
print("\nVacíos por columna predictora (las 15 con más, %):")
print((base[predictores].isna().mean().sort_values(ascending=False).head(15) * 100).round(1).to_string())

## Paso 7. Guardar el dataset final y su diccionario

**Por qué:** son los dos entregables de esta fase. El CSV tiene una fila por hogar: identificadores, ponderación, variable objetivo, variables solo de análisis y predictores, en ese orden. Se vuelve a leer para confirmar que no cambió nada (filas, vacíos ni ceros de los identificadores).

El diccionario trae, por columna: grupo (identificador, objetivo, predictor, solo análisis), tipo, tabla de origen, **significado y códigos oficiales del INEI**, cómo se construyó y porcentaje de vacíos. Para las variables creadas en el proyecto (por ejemplo `n_menores_5`), el significado es la definición con que se calcularon.

**Sobre abrirlo en Excel:** `id_hogar`, `conglome` y `ubigeo` son texto con ceros a la izquierda. Si Excel los muestra sin ceros, impórtalo como texto (Datos > Desde texto/CSV).

In [ ]:

orden = ([c for c in en_base if grupos[c] == "identificador"] + [c for c in en_base if grupos[c] == "ponderacion"]
         + [c for c in en_base if grupos[c] == "objetivo"] + [c for c in en_base if grupos[c] == "solo_analisis"]
         + numericas + binarias + categoricas)
assert set(orden) == set(en_base) and len(orden) == len(en_base)
dataset = base[orden].copy()

# Códigos enteros guardados como decimales (3.0) se escriben como enteros (3); los vacíos siguen vacíos.
for c in dataset.columns:
    if pd.api.types.is_float_dtype(dataset[c]):
        s = dataset[c].dropna()
        if len(s) and (s % 1 == 0).all():
            dataset[c] = dataset[c].astype("Int64")

ruta_csv = PROCESSED / "dataset_final.csv"
dataset.to_csv(ruta_csv, index=False, encoding="utf-8-sig")
lectura = pd.read_csv(ruta_csv, dtype={"id_hogar": str, "conglome": str, "ubigeo": str}, encoding="utf-8-sig")
assert lectura.shape == dataset.shape, "cambió el tamaño al leer el CSV"
assert (lectura["id_hogar"] == dataset["id_hogar"].values).all() and lectura["id_hogar"].str.len().eq(11).all()
assert (lectura.isna().sum() == dataset.isna().sum()).all(), "cambiaron los vacíos"
print(f"Dataset guardado y verificado: {ruta_csv} ({dataset.shape[0]:,} hogares x {dataset.shape[1]} columnas, {ruta_csv.stat().st_size / 1e6:.1f} MB)")

LABELS = {
 "p25_1": {
  "significado": "La calle, avenida,jirón, carretera, donde se ubica la vivienda tiene: Pista afaltada",
  "valores": "0 = Pase; 1 = Pista asfaltada"
 },
 "p25_2": {
  "significado": "La calle, avenida,jirón, carretera, donde se ubica la vivienda tiene: Pista afirmada/tierra",
  "valores": "0 = Pase; 1 = Pista afirmada/tierra"
 },
 "p25_3": {
  "significado": "La calle, avenida,jirón, carretera, donde se ubica la vivienda tiene: Veredas",
  "valores": "0 = Pase; 1 = Veredas"
 },
 "p25_4": {
  "significado": "La calle, avenida,jirón, carretera, donde se ubica la vivienda tiene: Poste alumbrado público",
  "valores": "0 = Pase; 1 = Poste alumbrado público"
 },
 "p25_5": {
  "significado": "La calle, avenida,jirón, carretera, donde se ubica la vivienda tiene: Ninguno",
  "valores": "0 = Pase; 1 = Ninguno"
 },
 "p1121": {
  "significado": "Tipo de alumbrado del hogar: Electricidad",
  "valores": "0 = Pase; 1 = Electricidad"
 },
 "p1123": {
  "significado": "Tipo de alumbrado del hogar: Petróleo/Gas (lámpara)",
  "valores": "0 = Pase; 1 = Petróleo/gas(Lámpara)"
 },
 "p1124": {
  "significado": "Tipo de alumbrado del hogar: Vela",
  "valores": "0 = Pase; 1 = Vela"
 },
 "p1125": {
  "significado": "Tipo de alumbrado del hogar: Generador",
  "valores": "0 = Pase; 1 = Generador"
 },
 "p1126": {
  "significado": "Tipo de alumbrado del hogar: Otro",
  "valores": "0 = Pase; 1 = Otro"
 },
 "p1127": {
  "significado": "No utiliza alumbrado en el hogar",
  "valores": "0 = Pase; 1 = No utiliza alumbrado en el hogar"
 },
 "p1131": {
  "significado": "Combustible que usan en el hogar para cocinar sus alimentos: Electricidad",
  "valores": "0 = Pase; 1 = Electricidad"
 },
 "p1132": {
  "significado": "Combustible que usan en el hogar para cocinar sus alimentos: Gas (balón GLP)",
  "valores": "0 = Pase; 1 = Gas (Balón GLP)"
 },
 "p1133": {
  "significado": "Combustible que usan en el hogar para cocinar sus alimentos: Gas Natural (Sistema de tuberías)",
  "valores": "0 = Pase; 1 = Gas Natural (sistema de tuberías)"
 },
 "p1135": {
  "significado": "Combustible que usan en el hogar para cocinar sus alimentos: Carbón",
  "valores": "0 = Pase; 1 = Carbón"
 },
 "p1136": {
  "significado": "Combustible que usan en el hogar para cocinar sus alimentos: Leña",
  "valores": "0 = Pase; 1 = Leña"
 },
 "p1139": {
  "significado": "Combustible que usan en el hogar para cocinar sus alimentos: Bosta, estiércol",
  "valores": "0 = Pase; 1 = Bosta, estiércol"
 },
 "p1137": {
  "significado": "Combustible que usan en el hogar para cocinar sus alimentos: Otro",
  "valores": "0 = Pase; 1 = Otro (Residuos agrícolas, etc)."
 },
 "p1138": {
  "significado": "No cocinan",
  "valores": "0 = Pase; 1 = No cocinan"
 },
 "p1141": {
  "significado": "Su hogar tiene: ¿Teléfono (fijo)?",
  "valores": "0 = Pase; 1 = Teléfono(fijo)"
 },
 "p1142": {
  "significado": "Su hogar tiene: ¿Teléfono Celular?",
  "valores": "0 = Pase; 1 = Teléfono Celular"
 },
 "p1143": {
  "significado": "Su hogar tiene: ¿Conexión a TV por cable o satelital?",
  "valores": "0 = Pase; 1 = Conexión a TV por cable o satelital"
 },
 "p1144": {
  "significado": "Su hogar tiene: ¿Conexión a Internet(fijo/Móvil)?",
  "valores": "0 = Pase; 1 = Conexión a Internet(fijo/móvil)"
 },
 "p1145": {
  "significado": "El hogar no tiene: Teléfono fijo, celular, TV cable, Internet",
  "valores": "0 = Pase; 1 = No tiene"
 },
 "p1146": {
  "significado": "Su hogar tiene: ¿Televisión Digital Terrestre?",
  "valores": "0 = Pase; 1 = Televisión Digital Terrestre"
 },
 "p24a": {
  "significado": "Las paredes exteriores de la vivienda tienen fachada: Con tarrajeo",
  "valores": "1 = Total; 2 = Parcial; 3 = No está tarrajeada; 4 = No corresponde; 9 = Missing Value"
 },
 "p24b": {
  "significado": "Las paredes exteriores de la vivienda tienen fachada: Pintada",
  "valores": "1 = Totalmente; 2 = Parcialmente; 3 = Sin pintar; 9 = Missing Value"
 },
 "p101": {
  "significado": "Tipo de vivienda",
  "valores": "1 = Casa independiente; 2 = Departamento en edificio; 3 = Vivienda en quinta; 4 = Vivienda en casa de vecindad (Callejón, solar o corralón); 5 = Choza o cabaña; 6 = Vivienda improvisada; 7 = Local no destinado para habitación humana; 8 = Otro"
 },
 "p102": {
  "significado": "El material predominante en las paredes exteriores es:",
  "valores": "1 = Ladrillo o bloque de cemento; 2 = Piedra o sillar con cal o cemento; 3 = Adobe; 4 = Tapia; 5 = Quincha (caña con barro); 6 = Piedra con barro; 7 = Madera (pona, tornillo, etc); 8 = Triplay/calamina/estera; 9 = Otro material"
 },
 "p103": {
  "significado": "El material predominante en los pisos es:",
  "valores": "1 = Parquet o madera pulida; 2 = Láminas asfálticas, vinílicos o similares; 3 = Losetas, terrazos o similares; 4 = Madera (pona, tornillo, etc); 5 = Cemento; 6 = Tierra; 7 = Otro material"
 },
 "p103a": {
  "significado": "El material predominante en los techos es:",
  "valores": "1 = Concreto armado; 2 = Madera; 3 = Tejas; 4 = Planchas de calamina, fibra de cemento o similares; 5 = Caña o estera con torta de barro o cemento; 6 = Triplay/estera/carrizo; 7 = Paja, hojas de palmera; 8 = Otro material"
 },
 "p105a": {
  "significado": "La vivienda que ocupa su hogar es:",
  "valores": "1 = Alquilada; 2 = Propia, totalmente pagada; 3 = Propia, por invasión; 4 = Propia, comprándola a plazos; 5 = Cedida por el centro de trabajo; 6 = Cedida por otro hogar o institución; 7 = Otra forma; 9 = Missing Value"
 },
 "p106a": {
  "significado": "¿Esta vivienda tiene título de propiedad?",
  "valores": "1 = Si; 2 = No; 3 = En trámite de titulación; 9 = Missing value"
 },
 "p106b": {
  "significado": "¿El título de la vivienda está Registrado en la SUNARP?",
  "valores": "1 = Si; 2 = No; 9 = Missing value"
 },
 "p110c": {
  "significado": "El hogar tiene acceso al servicio de agua todos los días de la semana",
  "valores": "1 = Si; 2 = No; 9 = Missing value"
 },
 "p112a": {
  "significado": "El servicio eléctrico de su hogar es:",
  "valores": "1 = Con medidor de uso exclusivo para la vivienda; 2 = Con medidor de uso colectivo (Para varias viviendas); 3 = Otro"
 },
 "p113a": {
  "significado": "Combustible que usan en el hogar para cocinar: Mayor Frecuencia",
  "valores": "1 = Electricidad; 2 = Gas (Balón GLP); 3 = Gas Natural (sistema de tuberias); 5 = Carbón; 6 = Leña; 7 = Otro; 8 = No cocinan"
 },
 "t110": {
  "significado": "(Re-codificado ) El abastecimiento de agua en su hogar procede de:",
  "valores": "1 = Red pública, dentro de la vivienda; 2 = Red pública, fuera de la vivienda pero dentro del edificio; 3 = Pilón o pileta de uso público; 4 = Camión - cisterna u otro similar; 5 = Pozo (agua subterránea); 6 = Manantial o puquio; 7 = Otra; 8 = Río, acequia, lago, laguna; 9 = Agua potable del vecino"
 },
 "t111a": {
  "significado": "Recodificada El baño o servicio higiénico que tiene su hogar",
  "valores": "1 = Red pública de desagüe dentro de la vivienda; 2 = Red pública de desagüe fuera de la vivienda pero dentro del edificio; 3 = Letrina (Con tratamiento); 4 = Pozo séptico, tanque séptico o biodigestor; 5 = Pozo ciego o negro; 6 = Río, acequia, canal o similar; 7 = Otra; 9 = Campo abierto o al aire libre; 10 = Letrina (Sin tratamiento); 11 = Letrina (Tipo compostera)"
 },
 "p104": {
  "significado": "¿Cuántas habitaciones en total tiene la vivienda, sin contar el baño,la cocina, los pasadizos, ni el garage?",
  "valores": "99 = Missing Value"
 },
 "p104a": {
  "significado": "¿Cuántas habitaciones se usan para exclusivamente para dormir?",
  "valores": "99 = Missing Value"
 },
 "p114b1": {
  "significado": "Conexión a internet a través de: Conexión fija (solo si el hogar tiene internet, P1144 = 1)",
  "valores": "0 = No aplica; 1 = Conexión fija"
 },
 "p114b2": {
  "significado": "Conexión a internet a través de: Conexión móvil pospago/control (solo si el hogar tiene internet, P1144 = 1)",
  "valores": "0 = No aplica; 1 = Conexión móvil pospago/control"
 },
 "p114b3": {
  "significado": "Conexión a internet a través de: Conexión móvil prepago (solo si el hogar tiene internet, P1144 = 1)",
  "valores": "0 = No aplica; 1 = Conexión móvil prepago"
 },
 "pobreza_cod": {
  "significado": "Condición de pobreza (INEI)",
  "valores": "1 = Pobre extremo; 2 = Pobre no extremo; 3 = No pobre"
 },
 "jefe_estado_civil": {
  "significado": "Estado civil o conyugal del jefe(a) (P209)",
  "valores": "Código P209 del diccionario ENAHO"
 },
 "jefe_lengua_materna": {
  "significado": "Idioma o lengua materna que aprendió en su niñez el jefe(a) (P300A)",
  "valores": "1 Quechua; 2 Aimara; 3 Otra lengua nativa; 4 Castellano; 6 Portugués; 7 Otra lengua extranjera; 8 No escucha/no habla; 9 Lengua de señas peruanas; 10 Ashaninka; 11 Awajún; 12 Shipibo-Konibo; 13 Shawi; 14 Matsigenka; 15 Achuar"
 }
}


def tipo(v):
    if v in numericas: return "numérica"
    if v in binarias: return "binaria (0/1)"
    if v in categoricas: return "categórica (código)"
    if v in ("id_hogar", "conglome", "ubigeo", "departamento"): return "texto"
    s = dataset[v].dropna()
    return "binaria (0/1)" if len(s) and set(s.unique()) <= {0, 1} else "numérica / código"


dicc["tipo"] = dicc["variable"].map(tipo)
dicc["significado"] = [LABELS.get(v, {}).get("significado", c) for v, c in zip(dicc["variable"], dicc["construccion"])]
dicc["codigos_o_valores"] = [LABELS.get(v, {}).get("valores", "") for v in dicc["variable"]]
dicc["vacios_pct"] = dicc["variable"].map(lambda v: round(dataset[v].isna().mean() * 100, 2))
dicc["posicion"] = dicc["variable"].map({v: i + 1 for i, v in enumerate(dataset.columns)})
dicc = dicc.sort_values("posicion")[["posicion", "variable", "grupo", "tipo", "origen", "significado", "codigos_o_valores", "construccion", "vacios_pct"]]
assert len(dicc) == dataset.shape[1]

dicc.to_csv(PROCESSED / "diccionario_dataset_final.csv", index=False, encoding="utf-8-sig")
try:
    with pd.ExcelWriter(PROCESSED / "diccionario_dataset_final.xlsx") as w:
        dicc.to_excel(w, sheet_name="Diccionario", index=False)
except ImportError:
    print("(sin openpyxl: solo se guardó el CSV del diccionario)")
print("Diccionario guardado:", len(dicc), "variables")
display(dicc)